# AI Leadership Insight Agent

This notebook demonstrates the AI Leadership Insight Agent, a Retrieval-Augmented Generation (RAG) system that answers leadership questions about company performance and status using internal documents.

## Overview

- **Document Ingestion**: Load and process company documents (PDFs and text files)
- **Chunking**: Split documents into manageable chunks
- **Embeddings**: Generate vector embeddings using OpenAI's text-embedding-3-small
- **Vector Store**: Store embeddings in FAISS for efficient retrieval
- **Retrieval**: Find relevant document chunks for user questions
- **Generation**: Use GPT-4o-mini to generate factual, structured answers
- **Visualization**: Plot chunk distributions and other insights

## Requirements

- Python 3.9+
- OpenAI API Key (set as environment variable `OPENAI_API_KEY`)
- Dependencies: `pip install -r requirements.txt`

In [ ]:
# Install and Import Libraries

# If running in Colab or fresh environment, uncomment and run:
# !pip install openai faiss-cpu numpy pypdf matplotlib

import os
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
import faiss
import matplotlib.pyplot as plt

# Import our custom modules
from config import *
from src.document_loader import load_documents
from src.chunking import chunk_text
from src.embeddings import get_embeddings
from src.vector_store import build_index
from src.retrieval import retrieve
from src.llm import generate_answer
from src.visualization import plot_chunk_distribution

print("Libraries imported successfully!")

In [ ]:
# Load and Process Documents

print("Loading documents from:", DATA_PATH)
docs = load_documents(DATA_PATH)
print(f"Loaded {len(docs)} documents")

# Display sample content from first document
print("\nSample from first document:")
print(docs[0][:200] + "...")

# Chunk the documents
print(f"\nChunking documents with size {CHUNK_SIZE} and overlap {CHUNK_OVERLAP}")
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_text(doc, CHUNK_SIZE, CHUNK_OVERLAP))

print(f"Created {len(all_chunks)} chunks")

# Visualize chunk distribution
plot_chunk_distribution(all_chunks)
print("Chunk distribution plot saved to outputs/chunk_distribution.png")

In [ ]:
# Create Embeddings for Documents

print(f"Creating embeddings using model: {EMBEDDING_MODEL}")
print("This may take a moment depending on the number of chunks...")

chunk_embeddings = get_embeddings(all_chunks, EMBEDDING_MODEL)
print(f"Generated embeddings for {len(chunk_embeddings)} chunks")
print(f"Embedding dimension: {len(chunk_embeddings[0])}")

In [ ]:
# Build Retrieval System

print("Building FAISS vector index...")
index = build_index(chunk_embeddings)
print("Vector index built successfully")

# Test retrieval with a sample question
sample_question = "What is our current revenue trend?"
print(f"\nTesting retrieval with question: '{sample_question}'")

question_embedding = get_embeddings([sample_question], EMBEDDING_MODEL)[0]
relevant_chunks = retrieve(question_embedding, index, all_chunks, TOP_K)

print(f"Retrieved {len(relevant_chunks)} relevant chunks:")
for i, chunk in enumerate(relevant_chunks, 1):
    print(f"\nChunk {i}:")
    print(chunk[:100] + "..." if len(chunk) > 100 else chunk)

In [ ]:
# Integrate LLM for Question Answering

print(f"LLM Model: {LLM_MODEL}")

# Function to answer a question
def answer_question(question):
    print(f"\nAnswering question: '{question}'")
    
    # Get question embedding
    question_embedding = get_embeddings([question], EMBEDDING_MODEL)[0]
    
    # Retrieve relevant chunks
    relevant_chunks = retrieve(question_embedding, index, all_chunks, TOP_K)
    context = "\n\n".join(relevant_chunks)
    
    # Generate answer
    answer = generate_answer(LLM_MODEL, context, question)
    
    print("Answer:")
    print(answer)
    return answer

# Test with sample question
sample_answer = answer_question(sample_question)

In [ ]:
# Test with Sample Questions

questions = [
    "What is our current revenue trend?",
    "Which risks were highlighted?",
    "What are the key strategic priorities?"
]

answers = []
for q in questions:
    ans = answer_question(q)
    answers.append((q, ans))
    print("-" * 80)

print("\nAll questions answered. Check outputs for visualizations.")

# Validation and Notes

## Assumptions Made

- Documents are in text (.txt) or PDF (.pdf) format
- OpenAI API key is available via environment variable
- Chunk size of 800 characters with 100 overlap is sufficient
- Top-4 retrieval provides adequate context for generation
- Structured prompt ensures consistent, factual responses

## Validation Approach

Since Adobe doesn't specify particular metrics, validation is performed through:

1. **Manual Inspection**: Review generated answers for factual accuracy and relevance
2. **Context Grounding**: Ensure answers are based only on provided documents
3. **Natural Language Quality**: Check for coherent, structured responses with appropriate sections
4. **Completeness**: Verify all aspects of questions are addressed

## Sample Validation Results

The agent successfully:
- Loads and processes documents
- Generates relevant embeddings
- Retrieves pertinent context
- Produces structured, document-grounded answers

## Potential Improvements

- Add more sophisticated chunking (sentence-aware)
- Implement reranking for better retrieval
- Add evaluation metrics (BLEU, ROUGE) for automated validation
- Support for more document formats
- Interactive question input interface

## Configuration

Models used:
- Embeddings: `text-embedding-3-small`
- Generation: `gpt-4o-mini`

API Key: Set `OPENAI_API_KEY` environment variable before running.